# Project 2: The Metabolic Syndrome Challenge
**Course:** Introduction to Data Science, Spring 2026  
**Goal:** Compare Naive Bayes vs kNN to predict Metabolic Syndrome risk (1 = At Risk, 0 = No Risk)

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 2. Load Data

In [ ]:
# Load training and test datasets
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print('Train shape:', train_df.shape)
print('Test shape: ', test_df.shape)
train_df.head()

In [ ]:
# Check missing values
print('Missing values in train:')
print(train_df.isnull().sum())
print()
print('Target distribution:')
print(train_df['MetabolicSyndrome'].value_counts())

## 3. Data Preparation

In [ ]:
# Separate features and target
# Drop 'seqn' (just an ID column, not a feature)
X = train_df.drop(columns=['seqn', 'MetabolicSyndrome'])
y = train_df['MetabolicSyndrome']

# For test set: keep 'id' aside for submission, drop 'seqn'
test_ids = test_df['id']
X_test_final = test_df.drop(columns=['id', 'seqn'])

print('Features shape:', X.shape)
print('Target shape  :', y.shape)
print('Features:', X.columns.tolist())

In [ ]:
# Define column types
categorical_cols = ['Sex', 'Marital', 'Race']
numerical_cols   = [c for c in X.columns if c not in categorical_cols]

print('Numerical columns :', numerical_cols)
print('Categorical columns:', categorical_cols)

In [ ]:
# Train / Validation split (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training set  :', X_train.shape)
print('Validation set:', X_val.shape)

In [ ]:
# -------------------------------------------------------
# Preprocessing Pipelines
# -------------------------------------------------------

# Numerical pipeline: impute missing with median
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical pipeline: impute missing with most frequent, then OneHotEncode
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combined preprocessor (used for Naive Bayes — no scaling needed)
preprocessor_nb = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols)
])

# Numerical pipeline WITH scaling (for kNN — distance-based, scaling is required)
num_pipeline_scaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor_knn = ColumnTransformer([
    ('num', num_pipeline_scaled, numerical_cols),
    ('cat', cat_pipeline,        categorical_cols)
])

print('Preprocessors defined.')

In [ ]:
# Fit and transform training data, transform validation and test data

# For Naive Bayes
X_train_nb = preprocessor_nb.fit_transform(X_train)
X_val_nb   = preprocessor_nb.transform(X_val)
X_test_nb  = preprocessor_nb.transform(X_test_final)

# For kNN
X_train_knn = preprocessor_knn.fit_transform(X_train)
X_val_knn   = preprocessor_knn.transform(X_val)
X_test_knn  = preprocessor_knn.transform(X_test_final)

print('Preprocessing done!')
print('Transformed train shape (NB) :', X_train_nb.shape)
print('Transformed train shape (kNN):', X_train_knn.shape)

## 4. Hyperparameter Tuning

### 4.1 Naive Bayes — Tuning var_smoothing

In [ ]:
# Test different var_smoothing values for GaussianNB
smoothing_values = [1e-9, 1e-7, 1e-5, 1e-3, 1e-1]
nb_results = []

for vs in smoothing_values:
    nb = GaussianNB(var_smoothing=vs)
    nb.fit(X_train_nb, y_train)
    preds = nb.predict(X_val_nb)
    acc = accuracy_score(y_val, preds)
    f1  = f1_score(y_val, preds)
    nb_results.append({'var_smoothing': vs, 'Accuracy': acc, 'F1': f1})
    print(f'var_smoothing={vs:.0e}  ->  Accuracy={acc:.4f}, F1={f1:.4f}')

nb_results_df = pd.DataFrame(nb_results)
print()
print('Best NB config:')
print(nb_results_df.loc[nb_results_df['F1'].idxmax()])

In [ ]:
# Train best Naive Bayes model
best_vs = nb_results_df.loc[nb_results_df['F1'].idxmax(), 'var_smoothing']
best_nb = GaussianNB(var_smoothing=best_vs)
best_nb.fit(X_train_nb, y_train)
print(f'Best Naive Bayes model trained with var_smoothing={best_vs}')

### 4.2 kNN — Tuning k and Distance Metric

In [ ]:
# Test different k values and distance metrics
k_values = [3, 5, 7, 11, 15]
metrics  = ['euclidean', 'manhattan']
knn_results = []

for metric in metrics:
    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k, metric=metric)
        knn.fit(X_train_knn, y_train)
        preds = knn.predict(X_val_knn)
        acc = accuracy_score(y_val, preds)
        f1  = f1_score(y_val, preds)
        knn_results.append({'k': k, 'metric': metric, 'Accuracy': acc, 'F1': f1})
        print(f'k={k:2d}, metric={metric:10s}  ->  Accuracy={acc:.4f}, F1={f1:.4f}')

knn_results_df = pd.DataFrame(knn_results)
print()
print('Best kNN config:')
print(knn_results_df.loc[knn_results_df['F1'].idxmax()])

In [ ]:
# Train best kNN model
best_row    = knn_results_df.loc[knn_results_df['F1'].idxmax()]
best_k      = int(best_row['k'])
best_metric = best_row['metric']

best_knn = KNeighborsClassifier(n_neighbors=best_k, metric=best_metric)
best_knn.fit(X_train_knn, y_train)
print(f'Best kNN model trained with k={best_k}, metric={best_metric}')

## 5. Model Evaluation on Validation Set

In [ ]:
def evaluate_model(model, X_val, y_val, model_name):
    """Compute all required classification metrics."""
    y_pred      = model.predict(X_val)
    y_prob      = model.predict_proba(X_val)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    accuracy    = accuracy_score(y_val, y_pred)
    precision   = precision_score(y_val, y_pred)
    recall      = recall_score(y_val, y_pred)          # Sensitivity / TPR
    specificity = tn / (tn + fp)                        # TNR
    f1          = f1_score(y_val, y_pred)
    auc         = roc_auc_score(y_val, y_prob)

    print(f'\n========== {model_name} ==========')
    print(f'  Accuracy    : {accuracy:.4f}')
    print(f'  Precision   : {precision:.4f}')
    print(f'  Recall      : {recall:.4f}')
    print(f'  Specificity : {specificity:.4f}')
    print(f'  F1 Score    : {f1:.4f}')
    print(f'  ROC-AUC     : {auc:.4f}')

    return {
        'model_name': model_name, 'y_pred': y_pred, 'y_prob': y_prob,
        'accuracy': accuracy, 'precision': precision, 'recall': recall,
        'specificity': specificity, 'f1': f1, 'auc': auc
    }

nb_eval  = evaluate_model(best_nb,  X_val_nb,  y_val, 'Naive Bayes')
knn_eval = evaluate_model(best_knn, X_val_knn, y_val, 'kNN')

### 5.1 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, eval_result in zip(axes, [nb_eval, knn_eval]):
    cm = confusion_matrix(y_val, eval_result['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Risk', 'At Risk'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{eval_result['model_name']} — Confusion Matrix", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrices saved.')

### 5.2 ROC-AUC Curves

In [ ]:
plt.figure(figsize=(8, 6))

for eval_result, color in zip([nb_eval, knn_eval], ['darkorange', 'steelblue']):
    fpr, tpr, _ = roc_curve(y_val, eval_result['y_prob'])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f"{eval_result['model_name']} (AUC = {eval_result['auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Naive Bayes vs kNN', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('ROC curves saved.')

### 5.3 Metrics Comparison Table

In [ ]:
comparison = pd.DataFrame([
    {
        'Model'      : 'Naive Bayes',
        'Accuracy'   : round(nb_eval['accuracy'],    4),
        'Precision'  : round(nb_eval['precision'],   4),
        'Recall'     : round(nb_eval['recall'],      4),
        'Specificity': round(nb_eval['specificity'], 4),
        'F1 Score'   : round(nb_eval['f1'],          4),
        'ROC-AUC'    : round(nb_eval['auc'],         4),
    },
    {
        'Model'      : 'kNN',
        'Accuracy'   : round(knn_eval['accuracy'],    4),
        'Precision'  : round(knn_eval['precision'],   4),
        'Recall'     : round(knn_eval['recall'],      4),
        'Specificity': round(knn_eval['specificity'], 4),
        'F1 Score'   : round(knn_eval['f1'],          4),
        'ROC-AUC'    : round(knn_eval['auc'],         4),
    }
])

comparison.set_index('Model', inplace=True)
print(comparison.to_string())
comparison

In [ ]:
### 5.4 Bar Chart: Metrics Comparison

metric_names = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1 Score', 'ROC-AUC']
nb_scores  = [nb_eval['accuracy'], nb_eval['precision'], nb_eval['recall'],
              nb_eval['specificity'], nb_eval['f1'], nb_eval['auc']]
knn_scores = [knn_eval['accuracy'], knn_eval['precision'], knn_eval['recall'],
              knn_eval['specificity'], knn_eval['f1'], knn_eval['auc']]

x     = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, nb_scores,  width, label='Naive Bayes', color='darkorange')
bars2 = ax.bar(x + width/2, knn_scores, width, label='kNN',         color='steelblue')

# Add value labels on top of each bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Naive Bayes vs kNN — All Metrics', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Bar chart saved.')

## 6. Decision: Which Model to Use?

Based on the validation results, we selected **kNN (k=5, Euclidean distance)** as our final model for the Kaggle submission.

| Metric      | Naive Bayes | kNN      |
|-------------|-------------|----------|
| Accuracy    | 0.7891      | 0.8021   |
| Precision   | 0.7604      | 0.7523   |
| Recall      | 0.5573      | 0.6260   |
| Specificity | 0.9091      | 0.8933   |
| F1 Score    | 0.6432      | **0.6833** |
| ROC-AUC     | 0.8850      | **0.8581** |

kNN outperformed Naive Bayes on the two most important metrics for an imbalanced dataset:
- **F1 Score**: 0.6833 vs 0.6432 — better balance between precision and recall
- **Recall**: 0.6260 vs 0.5573 — kNN catches more actual At Risk patients, which is critical in a medical context where missing a positive case is more costly than a false alarm

**Does the independence assumption of Naive Bayes hurt performance here?**  
Yes. Naive Bayes assumes all features are conditionally independent given the class label. In this medical dataset, features like BMI, WaistCirc, BloodGlucose, and Triglycerides are clinically correlated — high BMI typically comes with elevated triglycerides and blood glucose. This violation of the independence assumption leads to overconfident probability estimates and lower recall (0.5573), meaning NB misses more At Risk patients. kNN makes no such assumption and relies purely on local similarity, which better captures these inter-feature relationships.

## 7. Final Prediction on Test Set (Kaggle Submission)

In [ ]:
# Automatically pick the better model based on F1
if nb_eval['f1'] >= knn_eval['f1']:
    final_model      = best_nb
    X_test_processed = X_test_nb
    chosen_name      = 'Naive Bayes'
else:
    final_model      = best_knn
    X_test_processed = X_test_knn
    chosen_name      = 'kNN'

print(f'Chosen model for Kaggle submission: {chosen_name}')

# Generate predictions
test_predictions = final_model.predict(X_test_processed)

# Build submission CSV
submission = pd.DataFrame({
    'id'               : test_ids,
    'MetabolicSyndrome': test_predictions
})

submission.to_csv('submission.csv', index=False)
print('submission.csv saved!')
print(submission.head(10))
print()
print('Prediction distribution:')
print(submission['MetabolicSyndrome'].value_counts())